In [1]:
import pandas as pd
from pathlib import Path
from eutl_scraper import Settings

/Users/jan/git/eutl_scraper_v2/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [5]:
dir_extracted = Path("./data_tmp/extracted/")
settings = Settings(dir_data="data_tmp/")
fn_trans = settings.fp("transactions", settings.dir_extracted)
fn_inst = settings.fp("installations", settings.dir_extracted)
fn_accounts = settings.fp("accounts", settings.dir_extracted)
fn_holders = settings.fp("account_holders", settings.dir_extracted)
fn_compliance = settings.fp("compliance", settings.dir_extracted)
fn_projects = settings.fp("projects", settings.dir_extracted)
fn_trans = settings.fp("transactions", settings.dir_extracted)
fn_locations = settings.fp("installation_locations", settings.dir_extracted)
fn_nace_mapping = settings.fp("nace_from_leakage_lists", settings.dir_extracted)
fn_eex = settings.fp("eex_auctions", settings.dir_extracted)
fn_link_account_holder = settings.fp("link_account_holder", settings.dir_extracted)
fn_old_installation_data = Path(
    "manual_data/installations_euets_info_historical_batches.parquet"
)

## Missing installations

In [6]:
df_compliance = pd.read_csv(fn_compliance)
df_old_inst = pd.read_parquet(fn_old_installation_data)
df_inst = pd.read_csv(fn_inst)

inst_compl = set(df_compliance["installation_id"].unique())
inst_old = set(df_old_inst["id"].unique())
inst_new = set(df_inst["installation_id"].unique())

missing_new = inst_compl - inst_new
print(f"installations in compliance but not in new data: {len(missing_new)}")
in_old_data = missing_new.intersection(inst_old)
print(f"installations in compliance but not in new data, but in old data: {len(in_old_data)}")
for i in missing_new:
    if i in in_old_data:
        print(f"installation {i} is in old data")
    else:
        print(f"installation {i} is missing in both new and old data")

installations in compliance but not in new data: 0
installations in compliance but not in new data, but in old data: 0


In [7]:
for inst_id in list(missing_new):
    df_ = (
        df_compliance[df_compliance["installation_id"] == inst_id]
        .drop_duplicates(subset=["installation_name"])
    )
    if len(df_) > 1:
        print("!!!!Inconsistent!!!!")
        print(df_[["installation_id", "installation_name"]])
        continue
    print(df_.iloc[0]["installation_id"],df_.iloc[0]["installation_name"])

## Missing accounts 

In [10]:
df_accounts = pd.read_csv(fn_accounts, low_memory=False)
df_holders = pd.read_csv(fn_holders, low_memory=False)
df_link = pd.read_csv(fn_link_account_holder, low_memory=False)
df_trans = pd.read_csv(fn_trans, low_memory=False)

In [11]:
# get the missing account ids
linked_account_ids = set(df_link["account_id"])
linked_holder_ids = set(df_link["holder_id"])
account_ids = set(df_accounts["account_id"])
holder_ids = set(df_holders["holder_id"])
missing_account_ids = linked_account_ids - account_ids
missing_holder_ids = linked_holder_ids - holder_ids
print(f"Missing account ids: {missing_account_ids}")
print(f"Missing holder ids: {missing_holder_ids}")

Missing account ids: {'GB_5017316'}
Missing holder ids: set()


In [14]:
trans_parties = (
    set(df_trans.acquiring_account_id.unique())
    .union(
        set(df_trans.transferring_account_id.unique())
    )
)
missing_trans = linked_account_ids - trans_parties
print(f"Missing account ids in transactions: {len(missing_trans)}")
list(missing_trans)[:10]

Missing account ids in transactions: 13423


['LI_5016485',
 'DE_2537',
 'NL_5056652',
 'HU_5011770',
 'DE_5049630',
 'IT_5058799',
 'PL_5002624',
 'DE_5044914',
 'SK_5019157',
 'GB_5017357']

## Schemas 

In [25]:
df_ = pd.read_csv(fn_eex, low_memory=False)
for col in df_.columns:
    print(f'"{col}": "{col}",')

"auction_name": "auction_name",
"auction_price_eur_per_tco2": "auction_price_eur_per_tco2",
"auction_volume_tco2": "auction_volume_tco2",
"average_bid_size": "average_bid_size",
"average_bids_per_bidder": "average_bids_per_bidder",
"average_volume_bid_per_bidder": "average_volume_bid_per_bidder",
"average_volume_won_per_bidder": "average_volume_won_per_bidder",
"contract": "contract",
"country": "country",
"cover_ratio": "cover_ratio",
"date": "date",
"maximum_bid_eur_per_tco2": "maximum_bid_eur_per_tco2",
"mean_price_eur_per_tco2": "mean_price_eur_per_tco2",
"median_price_eur_per_tco2": "median_price_eur_per_tco2",
"minimum_bid_eur_per_tco2": "minimum_bid_eur_per_tco2",
"number_of_bids_submitted": "number_of_bids_submitted",
"number_of_successful_bidders": "number_of_successful_bidders",
"number_of_successful_bids": "number_of_successful_bids",
"revenue_at_eur": "revenue_at_eur",
"revenue_be_eur": "revenue_be_eur",
"revenue_bg_eur": "revenue_bg_eur",
"revenue_cy_eur": "revenue_cy_eur"

In [28]:
column_mapping = {
    "auction_name": "auction_name",
"auction_price_eur_per_tco2": "auction_price_eur_per_tco2",
"auction_volume_tco2": "auction_volume_tco2",
"average_bid_size": "average_bid_size",
"average_bids_per_bidder": "average_bids_per_bidder",
"average_volume_bid_per_bidder": "average_volume_bid_per_bidder",
"average_volume_won_per_bidder": "average_volume_won_per_bidder",
"contract": "contract",
"country": "country",
"cover_ratio": "cover_ratio",
"date": "date",
"maximum_bid_eur_per_tco2": "maximum_bid_eur_per_tco2",
"mean_price_eur_per_tco2": "mean_price_eur_per_tco2",
"median_price_eur_per_tco2": "median_price_eur_per_tco2",
"minimum_bid_eur_per_tco2": "minimum_bid_eur_per_tco2",
"number_of_bids_submitted": "number_of_bids_submitted",
"number_of_successful_bidders": "number_of_successful_bidders",
"number_of_successful_bids": "number_of_successful_bids",
"revenue_at_eur": "revenue_at_eur",
"revenue_be_eur": "revenue_be_eur",
"revenue_bg_eur": "revenue_bg_eur",
"revenue_cy_eur": "revenue_cy_eur",
"revenue_cz_eur": "revenue_cz_eur",
"revenue_de_eur": "revenue_de_eur",
"revenue_dk_eur": "revenue_dk_eur",
"revenue_ee_eur": "revenue_ee_eur",
"revenue_el_eur": "revenue_el_eur",
"revenue_es_eur": "revenue_es_eur",
"revenue_fi_eur": "revenue_fi_eur",
"revenue_fr_eur": "revenue_fr_eur",
"revenue_hr_eur": "revenue_hr_eur",
"revenue_hu_eur": "revenue_hu_eur",
"revenue_ie_eur": "revenue_ie_eur",
"revenue_innovation_fund_eur": "revenue_innovation_fund_eur",
"revenue_innovation_fund_rrf_eur": "revenue_innovation_fund_rrf_eur",
"revenue_is_eur": "revenue_is_eur",
"revenue_it_eur": "revenue_it_eur",
"revenue_li_eur": "revenue_li_eur",
"revenue_lt_eur": "revenue_lt_eur",
"revenue_lu_eur": "revenue_lu_eur",
"revenue_lv_eur": "revenue_lv_eur",
"revenue_modernisation_fund_eur": "revenue_modernisation_fund_eur",
"revenue_ms_rrf_eur": "revenue_ms_rrf_eur",
"revenue_mt_eur": "revenue_mt_eur",
"revenue_nl_eur": "revenue_nl_eur",
"revenue_no_eur": "revenue_no_eur",
"revenue_pl_eur": "revenue_pl_eur",
"revenue_pt_eur": "revenue_pt_eur",
"revenue_ro_eur": "revenue_ro_eur",
"revenue_se_eur": "revenue_se_eur",
"revenue_si_eur": "revenue_si_eur",
"revenue_sk_eur": "revenue_sk_eur",
"revenue_social_climate_fund_eur": "revenue_social_climate_fund_eur",
"standard_deviation_bid_volume_per_bidder": "standard_deviation_bid_volume_per_bidder",
"standard_deviation_volume_won_per_bidder": "standard_deviation_volume_won_per_bidder",
"status": "status",
"time": "time",
"total_amount_of_bids": "total_amount_of_bids",
"total_number_of_bidders": "total_number_of_bidders",
"total_revenue_eur": "total_revenue_eur",
"datetime": "datetime",
"created_at": "created_at",
}

df = (
    df_
    .assign(
        date = lambda x: pd.to_datetime(x["date"], utc=True),
        created_at = lambda x: pd.to_datetime(x["created_at"], utc=True),
        datetime = lambda x: pd.to_datetime(x["datetime"], utc=True),
    )
    .rename(columns=column_mapping)
)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 62 columns):
 #   Column                                    Non-Null Count  Dtype              
---  ------                                    --------------  -----              
 0   auction_name                              57 non-null     str                
 1   auction_price_eur_per_tco2                57 non-null     float64            
 2   auction_volume_tco2                       57 non-null     int64              
 3   average_bid_size                          57 non-null     float64            
 4   average_bids_per_bidder                   57 non-null     float64            
 5   average_volume_bid_per_bidder             57 non-null     float64            
 6   average_volume_won_per_bidder             57 non-null     float64            
 7   contract                                  57 non-null     str                
 8   country                                   57 non-null     str            

In [23]:
df[df["installation_id"].duplicated(keep=False)]

,installation_id,nace_2015,nace_2020
1862,DE_1486,35.11,35.11
1863,DE_1486,35.11,35.11
3528,DE_74,23.51,23.51
3529,DE_74,23.51,23.51
3540,DE_75,23.51,23.51
3541,DE_75,23.51,23.51
3823,DK_11,35.30,35.30
3824,DK_11,35.30,35.30
3835,DK_12,35.30,35.30
3836,DK_12,35.30,35.30
